<h1 style=\"text-align: center; font-size: 50px;\"> Register Model </h1>

# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Register the Model Log Results to MLFlow

# Start Execution

In [1]:
import os
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

2025-09-09 14:00:22 - INFO - Notebook execution started.


# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 148 ms, sys: 73.6 ms, total: 221 ms
Wall time: 5.43 s


In [9]:
# ------------------------- Import Services -------------------------

import tempfile
import shutil
import sys

import mlflow
import mlflow.pyfunc
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, TensorSpec, ParamSchema, ParamSpec
from mlflow.tracking import MlflowClient

# # Define the relative path to the 'src' directory (two levels up from current working directory)
src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add 'src' directory to system path for module imports (e.g., utils)
if src_path not in sys.path:
    sys.path.append(src_path)

# Import new MLflow models-from-code components
from src.mlflow import Logger
from src.utils import (
    load_config,
)

src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if src_path not in sys.path:
    sys.path.append(src_path)

# Correct import for onnx_utils
from src.onnx_utils import ModelExportConfig

# Configure Settings

In [5]:
# ------------------------- Model File Paths -------------------------
MT_MODEL = "Helsinki-NLP/opus-mt-en-es"
ASR_MODEL_PATH = "/home/jovyan/datafabric/STT_En_Citrinet_1024_Gamma_0.25/stt_en_citrinet_1024_gamma_0_25.nemo"                  # Speech-to-Text (ASR) model
SPECTROGRAM_GENERATOR_PATH = "/home/jovyan/datafabric/TTS_Es_Multispeaker_FastPitch_HiFiGAN/tts_es_fastpitch_multispeaker.nemo"  # Spectrogram generator model (FastPitch)
VOCODER_PATH = "/home/jovyan/datafabric/TTS_Es_Multispeaker_FastPitch_HiFiGAN/tts_es_hifigan_ft_fastpitch_multispeaker.nemo"     # Vocoder model (HiFiGAN)
CONFIG_PATH = "../configs/config.yaml"
AUDIO_SAMPLE_PATH = "../data/ForrestGump.mp3"      # Path to the input English audio sample

# ------------------------- MLflow Experiment Configuration -------------------------

EXPERIMENT_NAME = "NeMo_Translation_Experiment"    # MLflow experiment name
RUN_NAME = "NeMo_en_es_Translation_Run"            # Specific run name inside the experiment
MODEL_NAME = "nemo_en_es"                          # Registered model name in MLflow
DEMO_PATH = "../demo"                              # Path to save demo outputs

In [6]:
# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")

✅ Configuration loaded successfully


In [ ]:


# ------------------------- Helper Functions -------------------------

def create_nemo_models_dict():
    """Create NeMo models dictionary from defined paths."""
    return {
        "enc_dec_CTC": ASR_MODEL_PATH,
        "fast_pitch": SPECTROGRAM_GENERATOR_PATH,
        "hifi_gan": VOCODER_PATH
    }

def create_sample_signature():
    """Create MLflow model signature for NeMo Audio Translation."""
    import mlflow
    from mlflow.types.schema import Schema, ColSpec
    import pandas as pd
    
    # Input schema for audio translation
    input_schema = Schema([
            ColSpec("string", "source_text"),
            ColSpec("string", "source_serialized_audio"),
        ])

    output_schema = Schema([
        ColSpec("string", "original_text"),
        ColSpec("string", "translated_text"),
        ColSpec("string", "translated_serialized_audio"),
    ])

    params_schema = ParamSchema([
        ParamSpec("use_audio", "boolean", False)
    ])
    
    return ModelSignature(
            inputs=input_schema,
            outputs=output_schema,
            params=params_schema
        )

def load_models_for_onnx_conversion():
    """
    Load all models into memory for ONNX conversion.
    
    Returns:
        tuple: (nemo_models_dict, loaded_models_dict)
    """
    import torch
    import nemo.collections.asr as nemo_asr
    import nemo.collections.tts as nemo_tts
    from transformers import MarianMTModel, MarianTokenizer
    
    nemo_models = create_nemo_models_dict()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load all models into memory
    loaded_models = {
        'mt_model': MarianMTModel.from_pretrained(MT_MODEL),
        'asr_model': nemo_asr.models.EncDecCTCModel.restore_from(nemo_models["enc_dec_CTC"]),
        'fast_pitch_model': nemo_tts.models.FastPitchModel.restore_from(nemo_models["fast_pitch"]),
        'hifi_gan_model': nemo_tts.models.HifiGanModel.restore_from(nemo_models["hifi_gan"])
    }
    
    logger.info("All models loaded into memory for ONNX conversion")
    return nemo_models, loaded_models

def create_and_convert_onnx_models(loaded_models):
    """
    Create ONNX versions of all models and save them to a temporary directory.
    
    Args:
        loaded_models: Dictionary of loaded model objects
    
    Returns:
        str: Path to directory containing ONNX model files
    """
    import torch
    from optimum.onnxruntime import ORTModelForSeq2SeqLM
    from optimum.exporters.onnx import main_export
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create temp directory for ONNX models
    onnx_dir = os.path.join(tempfile.gettempdir(), "onnx_models")
    if os.path.exists(onnx_dir):
        shutil.rmtree(onnx_dir)
    os.makedirs(onnx_dir)
    
    try:
        # Convert Transformers model to ONNX
        mt_model = loaded_models['mt_model']
        mt_onnx_path = os.path.join(onnx_dir, "Helsinki-NLP.onnx")
        
        # Export using optimum
        dummy_input = {"input_ids": torch.randint(0, 1000, (1, 10))}
        torch.onnx.export(
            mt_model,
            dummy_input,
            mt_onnx_path,
            export_params=True,
            opset_version=12,
            do_constant_folding=True,
            input_names=['input_ids'],
            output_names=['output'],
            dynamic_axes={'input_ids': {0: 'batch_size', 1: 'sequence'},
                         'output': {0: 'batch_size', 1: 'sequence'}}
        )
        logger.info(f"Converted Helsinki-NLP to ONNX: {mt_onnx_path}")
        
        # Convert NeMo models to ONNX using their built-in export
        nemo_models_info = [
            ('asr_model', 'enc_dec_CTC.onnx'),
            ('fast_pitch_model', 'fast_pitch.onnx'), 
            ('hifi_gan_model', 'hifi_gan.onnx')
        ]
        
        for model_key, onnx_filename in nemo_models_info:
            model = loaded_models[model_key].to(device)
            onnx_path = os.path.join(onnx_dir, onnx_filename)
            
            # Use NeMo's built-in ONNX export
            model.export(onnx_path, check_trace=False)
            logger.info(f"Converted {model_key} to ONNX: {onnx_path}")
            
        logger.info(f"All models converted to ONNX in directory: {onnx_dir}")
        return onnx_dir
        
    except Exception as e:
        logger.error(f"Error during ONNX conversion: {str(e)}")
        logger.info("Continuing without ONNX conversion...")
        # Return empty directory if conversion fails
        return onnx_dir

def prepare_models_with_onnx(nemo_models, onnx_dir):
    """
    Prepare models directory containing both .nemo and .onnx files.
    
    Args:
        nemo_models: Dictionary of NeMo model paths
        onnx_dir: Directory containing ONNX model files
    
    Returns:
        str: Path to directory containing all model files
    """
    models_dir = os.path.join(tempfile.gettempdir(), "all_models")
    if os.path.exists(models_dir):
        shutil.rmtree(models_dir)
    os.makedirs(models_dir)
    
    # Copy NeMo models
    for model_name, model_path in nemo_models.items():
        if os.path.exists(model_path):
            target_filename = f"{model_name}.nemo"
            shutil.copy2(model_path, os.path.join(models_dir, target_filename))
            logger.info(f"Copied NeMo model: {model_name} -> {target_filename}")
    
    # Copy ONNX models if they exist
    if os.path.exists(onnx_dir):
        for onnx_file in os.listdir(onnx_dir):
            if onnx_file.endswith('.onnx'):
                src_path = os.path.join(onnx_dir, onnx_file)
                dst_path = os.path.join(models_dir, onnx_file)
                shutil.copy2(src_path, dst_path)
                logger.info(f"Copied ONNX model: {onnx_file}")
    
    return models_dir

def create_dummy_docs_directory():
    """Create minimal docs directory for vanilla-rag compatibility."""
    temp_docs_dir = os.path.join(tempfile.gettempdir(), "dummy_docs")
    if os.path.exists(temp_docs_dir):
        shutil.rmtree(temp_docs_dir)
    os.makedirs(temp_docs_dir)
    
    # Create a dummy file
    dummy_file = os.path.join(temp_docs_dir, "README.md")
    with open(dummy_file, 'w') as f:
        f.write("# NeMo Audio Translation Model\n\nThis model uses NeMo for audio translation with ONNX support.")
    
    return temp_docs_dir

# ------------------------- Registration Function -------------------------

def register_nemo_translation_model():
    """Register NeMo Audio Translation model with ONNX conversion using vanilla-rag Logger."""
    
    logger.info("Starting NeMo Audio Translation model registration with ONNX conversion...")
    
    # Load all models and convert to ONNX
    nemo_models, loaded_models = load_models_for_onnx_conversion()
    
    # Convert models to ONNX format
    onnx_dir = create_and_convert_onnx_models(loaded_models)
    
    # Prepare combined models directory (both .nemo and .onnx)
    models_dir = prepare_models_with_onnx(nemo_models, onnx_dir)
    docs_dir = create_dummy_docs_directory()
    
    # Create model signature
    signature = create_sample_signature()
    logger.info("Created MLflow model signature")
    
    try:
        # Register model using vanilla-rag Logger interface
        # Both .nemo and .onnx files will be in the models/ subdirectory
        Logger.log_model(
            signature=signature,
            artifact_path=MODEL_NAME,
            config_path=CONFIG_PATH,
            docs_path=docs_dir,      # Dummy docs for interface compliance
            model_path=models_dir,   # Combined .nemo and .onnx models -> /artifacts/data/models/
            demo_folder="../demo"
        )
        
        logger.info(f"✅ NeMo Audio Translation model '{MODEL_NAME}' registered successfully with ONNX support!")
    
    finally:
        # Clean up temporary directories
        temp_dirs = [models_dir, docs_dir, onnx_dir]
        for temp_dir in temp_dirs:
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
                logger.info(f"Cleaned up temporary directory: {temp_dir}")

# ------------------------- Execute Registration -------------------------

register_nemo_translation_model()

2025-09-09 14:10:51 - INFO - Starting NeMo Audio Translation model registration with ONNX conversion...


In [ ]:
# ------------------------- Success Confirmation -------------------------

print(f"✅ Model '{MODEL_NAME}' successfully logged and registered under experiment '{EXPERIMENT_NAME}'.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).